# LS-CP-FLCB

## Аппроксимация ограниченного Парето-фронта в функциональных многоруких бандитах

Этот ноутбук содержит теоретическую основу алгоритма **LS-CP-FLCB** — *Lipschitz-Shared Constrained Pareto Functional Lower Confidence Bound*.

Основная идея алгоритма состоит в следующем. Каждая ручка рассматривается как функциональное семейство с внутренними параметрами. У каждой ручки есть несколько целевых функций и несколько ограничений. Главная цель состоит не только в идентификации релевантных ручек, а в аппроксимации ограниченного Парето-фронта. При этом близкие направления скаляризации должны обмениваться информацией, поскольку при гладком фронте близкие направления обычно соответствуют близким оптимальным решениям.


## 1. Интуитивная мотивация

В классической задаче MAB имеется конечное число ручек, и на каждом шаге алгоритм выбирает одну из них. После выбора он получает случайную награду или потерю. Такая модель полезна, когда каждая альтернатива является фиксированным объектом.

В задаче FMAB каждая ручка является не одной случайной величиной, а целым функциональным семейством. Например, ручка может соответствовать классу моделей `SmallCNN`, `ResNet`, `MobileNet` или некоторому методу обучения. Внутри каждой ручки выбирается параметр

$$
x\in X_i,
$$

где $X_i$ — пространство внутренних параметров ручки $i$.

В задачах машинного обучения параметр $x$ может включать скорость обучения, коэффициент регуляризации, dropout, ширину модели, число эпох обучения и другие гиперпараметры.


## 2. Многокритериальная постановка

В реальных задачах выбора модели обычно важна не одна цель, а несколько. Например, модель может оцениваться по ошибке на валидации, задержке инференса, размеру модели, потреблению памяти и времени обучения.

Поэтому для каждой ручки $i$ вводится вектор целей

$$
Y_i(x)=\bigl(y_{i,1}(x),\dots,y_{i,D}(x)\bigr)\in\mathbb{R}^D.
$$

Все критерии считаются минимизируемыми. Если некоторый критерий нужно максимизировать, он заменяется на противоположный

Также вводится вектор ограничений

$$
C_i(x)=\bigl(c_{i,1}(x),\dots,c_{i,Q}(x)\bigr)\in\mathbb{R}^Q.
$$

Ограничения имеют вид

$$
C_i(x)\le B,
$$

то есть покоординатно

$$
c_{i,q}(x)\le B_q,\qquad q=1,\dots,Q.
$$


## 3. Допустимые множества

Для каждой ручки $i$ определим допустимое множество внутренних параметров

$$
F_i=\{x\in X_i: C_i(x)\le B\}.
$$

Здесь $F_i$ может быть пустым, если ручка $i$ не имеет допустимых конфигураций.

Допустимое множество значений целей для ручки $i$ определяется как

$$
\mathcal{Y}_i^{\mathrm{feas}}=\{Y_i(x):x\in F_i\}.
$$

Общее допустимое множество значений целей по всем ручкам:

$$
\mathcal{Y}^{\mathrm{feas}}=\bigcup_{i=1}^K \mathcal{Y}_i^{\mathrm{feas}}.
$$

Именно на этом множестве строится ограниченный Парето-фронт.


## 4. Парето-доминирование

Пусть $a,b\in\mathbb{R}^D$ — два вектора целей. Поскольку все критерии минимизируются, точка $a$ доминирует точку $b$, если

$$
a_d\le b_d\quad \text{для всех } d=1,\dots,D,
$$

и существует хотя бы один индекс $d_0\in\{1,\dots,D\}$, такой что

$$
a_{d_0}<b_{d_0}.
$$

Множество недоминируемых точек множества $\mathcal{A}$ обозначается

$$
\operatorname{ND}(\mathcal{A}).
$$

Истинный ограниченный **Pareto front** (Парето-фронт) определяется как

$$
\mathcal{P}_{\mathrm{feas}}^\star=\operatorname{ND}\left(\mathcal{Y}^{\mathrm{feas}}\right).
$$

Главная цель алгоритма — построить хорошее приближение множества $\mathcal{P}_{\mathrm{feas}}^\star$ за конечный бюджет обращений к оракулам.


## 5. Почему главная цель — аппроксимация фронта

Можно определить множество **CP-relevant arms** (*constrained Pareto-relevant arms*, ограниченно Парето-релевантные ручки):

$$
I_{\mathrm{CP}}^\star=\left\{i:\mathcal{Y}_i^{\mathrm{feas}}\cap\mathcal{P}_{\mathrm{feas}}^\star\ne\varnothing\right\}.
$$

Однако эта цель может быть слабой. В прикладных задачах возможно, что почти каждое семейство моделей имеет хотя бы одну недоминируемую допустимую конфигурацию. Тогда

$$
I_{\mathrm{CP}}^\star=\{1,\dots,K\},
$$

и задача идентификации ручек становится малоинформативной.

Поэтому в данной версии основной целью считается не только нахождение $I_{\mathrm{CP}}^\star$, а восстановление самого ограниченного Парето-фронта:

$$
\widehat{\mathcal{P}}_T\approx \mathcal{P}_{\mathrm{feas}}^\star.
$$

Множество релевантных ручек после этого можно рассматривать как побочный интерпретируемый результат.


## 6. Выход алгоритма

Пусть за бюджет $T$ алгоритм получил наблюдения

$$
(i_t,x_t,Y_{i_t}(x_t),C_{i_t}(x_t)),\qquad t=1,\dots,T.
$$

Множество найденных допустимых точек:

$$
\mathcal{S}_T^{\mathrm{feas}}=\{Y_{i_t}(x_t):t\le T,\ C_{i_t}(x_t)\le B\}.
$$

Найденный Парето-фронт:

$$
\widehat{\mathcal{P}}_T=\operatorname{ND}\left(\mathcal{S}_T^{\mathrm{feas}}\right).
$$

Дополнительно можно определить найденные релевантные ручки:

$$
\widehat{I}_T=\left\{i:\exists x\in X_i\ \text{такой, что}\ Y_i(x)\in\widehat{\mathcal{P}}_T\right\}.
$$

В дальнейшем основной объект анализа — $\widehat{\mathcal{P}}_T$.


## 7. Метрики качества

### Hypervolume gap

**Hypervolume** (гиперобъем) множества точек относительно reference point (опорной точки) $r$ обозначается $HV(\cdot)$. Для задачи минимизации опорная точка выбирается хуже всех рассматриваемых точек.

Hypervolume gap:

$$
HV\text{-gap}(T)=HV(\mathcal{P}_{\mathrm{feas}}^\star)-HV(\widehat{\mathcal{P}}_T).
$$

Чем меньше эта величина, тем лучше.

### Hypervolume ratio

$$
HV\text{-ratio}(T)=\frac{HV(\widehat{\mathcal{P}}_T)}{HV(\mathcal{P}_{\mathrm{feas}}^\star)}.
$$

Если $HV\text{-ratio}=0.95$, то найденный фронт покрывает 95% гиперобъема истинного фронта.

### IGD

**IGD** — *Inverted Generational Distance* (обратное порождающее расстояние):

$$
IGD(T)=\frac{1}{|\mathcal{P}_{\mathrm{feas}}^\star|}\sum_{p\in\mathcal{P}_{\mathrm{feas}}^\star}\min_{\hat p\in\widehat{\mathcal{P}}_T}\|p-\hat p\|.
$$

Чем меньше IGD, тем лучше истинный фронт покрыт найденными точками.


## 8. Скаляризация направлений

Для аппроксимации Парето-фронта вводится конечный набор направлений

$$
\Lambda=\{\lambda_1,\dots,\lambda_L\},
$$

где каждое направление лежит на симплексе

$$
\lambda_\ell\in\Delta^{D-1}.
$$

Это означает

$$
\lambda_{\ell,d}\ge 0,\qquad \sum_{d=1}^D\lambda_{\ell,d}=1.
$$

Направление $\lambda_\ell$ описывает один компромисс между критериями. Например, при двух критериях направление $(0.9,0.1)$ делает больший акцент на первом критерии, а направление $(0.1,0.9)$ — на втором.


## 9. Скаляризация Чебышева

Используется **augmented Chebyshev scalarization** (дополненная скаляризация Чебышева):

$$
s_\lambda(y)=\max_{d=1,\dots,D}\lambda_d(y_d-z_d)+\rho\sum_{d=1}^D\lambda_d(y_d-z_d).
$$

Здесь:

- $y\in\mathbb{R}^D$ — вектор целей;
- $z\in\mathbb{R}^D$ — **ideal point** (идеальная точка);
- $\lambda\in\Delta^{D-1}$ — направление;
- $\rho>0$ — малый коэффициент, например $\rho=10^{-3}$.

Для ручки $i$ и направления $\lambda_\ell$ определим оптимальное допустимое скаляризованное значение

$$
\phi_{i,\ell}^\star=\inf_{x\in X_i:C_i(x)\le B}s_{\lambda_\ell}(Y_i(x)).
$$

Лучшее значение по направлению $\ell$ среди всех ручек:

$$
\phi_\ell^\star=\min_{i=1,\dots,K}\phi_{i,\ell}^\star.
$$

Скаляризованный зазор ячейки $(i,\ell)$:

$$
\Delta_{i,\ell}=\phi_{i,\ell}^\star-\phi_\ell^\star.
$$


## 10. Ячейки arm-direction

Алгоритм работает с ячейками

$$
(i,\ell),
$$

где $i$ — номер ручки, а $\ell$ — номер направления. Ячейка $(i,\ell)$ означает, что ручка $i$ рассматривается как кандидат на оптимальность в направлении $\lambda_\ell$.

Для каждой ячейки хранится счетчик прямых запусков

$$
n_{i,\ell}(t)=\#\{\tau<t:i_\tau=i,\ \ell_\tau=\ell\}.
$$

Если $K$ — число ручек, а $L$ — число направлений, то всего имеется $KL$ ячеек. В реализации это соответствует матрицам размера $K\times L$.


## 11. Совместное использование наблюдений

Близкие направления скаляризации часто соответствуют близким областям Парето-фронта, поэтому для каждой ручки $i$ храним общий набор наблюдений

$$
D_i(t)=\{(x_s,Y_i(x_s),C_i(x_s)): \text{ручка }i\text{ была запущена до момента }t\}.
$$

Каждое новое наблюдение ручки $i$ используется для всех направлений. Для направления $\ell$ оценка лучшего найденного допустимого значения равна

$$
\widehat{\phi}_{i,\ell}(t)=\min_{(x,y,c)\in D_i(t):c\le B}s_{\lambda_\ell}(y).
$$

Если допустимых точек в $D_i(t)$ нет, то

$$
\widehat{\phi}_{i,\ell}(t)=+\infty.
$$


## 12. Предположения

### A1. Ограниченность целей

Существует константа $R>0$, такая что для всех $i$, $x$ и $d$:

$$
0\le Y_{i,d}(x)-z_d\le R.
$$

### A2. Допустимые множества

Для каждой ручки множество

$$
F_i=\{x\in X_i:C_i(x)\le B\}
$$

является компактным или пустым.

### A3. Сходимость внутреннего оптимизатора

При прямых запусках ячейки $(i,\ell)$ ошибка внутренней оптимизации удовлетворяет

$$
\gamma_{i,\ell}(n)=a_{i,\ell}n^{-\alpha_{i,\ell}},
$$

где $a_{i,\ell}>0$ и $\alpha_{i,\ell}>0$.

### A4. Корректность локальных доверительных оценок

С вероятностью не меньше $1-\delta$ для всех $i$, $\ell$ и $t$ выполнено

$$
L_{i,\ell}^{\mathrm{loc}}(t)\le \phi_{i,\ell}^\star\le U_{i,\ell}^{\mathrm{loc}}(t).
$$

Это событие обозначим через $\mathcal{E}$ и будем называть благоприятным событием.


## 13. Локальные доверительные оценки

Пусть $b_{i,\ell}(t,\delta)$ — statistical confidence radius (статистический доверительный радиус). Например, для субгауссовского шума он может иметь порядок

$$
b_{i,\ell}(t,\delta)=O\left(\sqrt{\frac{\log(KLt^2/\delta)}{\max(1,n_{i,\ell}(t))}}\right).
$$

Пусть $\widehat{\phi}_{i,\ell}^{\mathrm{dir}}(t)$ — лучшее допустимое значение, полученное прямыми запусками ячейки $(i,\ell)$.

Локальная нижняя доверительная граница:

$$
L_{i,\ell}^{\mathrm{loc}}(t)=\widehat{\phi}_{i,\ell}^{\mathrm{dir}}(t)-\gamma_{i,\ell}(n_{i,\ell}(t))-b_{i,\ell}(t,\delta).
$$

Локальная верхняя доверительная граница:

$$
U_{i,\ell}^{\mathrm{loc}}(t)=\widehat{\phi}_{i,\ell}(t)+b_{i,\ell}(t,\delta).
$$


## 14. Лемма 1: липшицевость скаляризации Чебышева по направлению

### Формулировка

Пусть выполнено предположение A1. Пусть $\lambda,\lambda'\in\Delta^{D-1}$ — два направления. Тогда для любого $y$, удовлетворяющего $0\le y_d-z_d\le R$, выполнено

$$
|s_\lambda(y)-s_{\lambda'}(y)|\le (1+\rho)R\|\lambda-\lambda'\|_1.
$$

Обозначим

$$
L_s=(1+\rho)R.
$$

Тогда

$$
|s_\lambda(y)-s_{\lambda'}(y)|\le L_s\|\lambda-\lambda'\|_1.
$$


## 15. Доказательство леммы 1

Введем обозначение

$$
a_d=y_d-z_d,\qquad d=1,\dots,D.
$$

По предположению A1 имеем

$$
0\le a_d\le R.
$$

Определим первую часть скаляризации:

$$
g_\lambda(y)=\max_d \lambda_d a_d.
$$

Тогда

$$
|g_\lambda(y)-g_{\lambda'}(y)|=\left|\max_d \lambda_d a_d-\max_d \lambda'_d a_d\right|.
$$

Используем неравенство

$$
\left|\max_d u_d-\max_d v_d\right|\le \max_d |u_d-v_d|.
$$

Положим

$$
u_d=\lambda_d a_d,\qquad v_d=\lambda'_d a_d.
$$

Тогда

$$
|g_\lambda(y)-g_{\lambda'}(y)|\le \max_d |(\lambda_d-\lambda'_d)a_d|.
$$

Так как $a_d\le R$, получаем

$$
|g_\lambda(y)-g_{\lambda'}(y)|\le R\|\lambda-\lambda'\|_\infty\le R\|\lambda-\lambda'\|_1.
$$

Теперь рассмотрим вторую часть скаляризации:

$$
h_\lambda(y)=\rho\sum_d\lambda_d a_d.
$$

Тогда

$$
|h_\lambda(y)-h_{\lambda'}(y)|=\rho\left|\sum_d(\lambda_d-\lambda'_d)a_d\right|.
$$

По неравенству треугольника:

$$
|h_\lambda(y)-h_{\lambda'}(y)|\le \rho\sum_d |\lambda_d-\lambda'_d|a_d.
$$

Так как $a_d\le R$, имеем

$$
|h_\lambda(y)-h_{\lambda'}(y)|\le \rho R\|\lambda-\lambda'\|_1.
$$

Складывая оценки для $g_\lambda$ и $h_\lambda$, получаем

$$
|s_\lambda(y)-s_{\lambda'}(y)|\le (1+\rho)R\|\lambda-\lambda'\|_1.
$$

Лемма доказана.


## 16. Следствие 1: липшицевость оптимальных скаляризованных значений

### Формулировка

Пусть

$$
\phi_i^\star(\lambda)=\inf_{x\in X_i:C_i(x)\le B}s_\lambda(Y_i(x)).
$$

Тогда для любых $\lambda,\lambda'\in\Delta^{D-1}$ выполнено

$$
|\phi_i^\star(\lambda)-\phi_i^\star(\lambda')|\le L_s\|\lambda-\lambda'\|_1.
$$

### Доказательство

Обозначим

$$
d_{\lambda,\lambda'}=\|\lambda-\lambda'\|_1.
$$

По Лемме 1 для любого допустимого $x\in F_i$:

$$
|s_\lambda(Y_i(x))-s_{\lambda'}(Y_i(x))|\le L_s d_{\lambda,\lambda'}.
$$

Следовательно,

$$
s_\lambda(Y_i(x))\le s_{\lambda'}(Y_i(x))+L_s d_{\lambda,\lambda'}.
$$

Берем инфимум по $x\in F_i$:

$$
\phi_i^\star(\lambda)\le \phi_i^\star(\lambda')+L_s d_{\lambda,\lambda'}.
$$

Меняя местами $\lambda$ и $\lambda'$, получаем

$$
\phi_i^\star(\lambda')\le \phi_i^\star(\lambda)+L_s d_{\lambda,\lambda'}.
$$

Два последних неравенства дают

$$
|\phi_i^\star(\lambda)-\phi_i^\star(\lambda')|\le L_s d_{\lambda,\lambda'}.
$$

Следствие доказано.


## 17. Разделяемые доверительные оценки

Обозначим расстояние между направлениями $\lambda_\ell$ и $\lambda_m$:

$$
d_{\ell m}=\|\lambda_\ell-\lambda_m\|_1.
$$

Разделяемая нижняя доверительная граница:

$$
L_{i,\ell}^{\mathrm{sh}}(t)=\max_{m=1,\dots,L}\left[L_{i,m}^{\mathrm{loc}}(t)-L_s d_{\ell m}\right].
$$

Разделяемая верхняя доверительная граница:

$$
U_{i,\ell}^{\mathrm{sh}}(t)=\min_{m=1,\dots,L}\left[U_{i,m}^{\mathrm{loc}}(t)+L_s d_{\ell m}\right].
$$

Здесь индекс $m$ обозначает вспомогательное направление, из которого переносится информация в целевое направление $\ell$.


## 18. Лемма 2: корректность разделяемых границ

### Формулировка

Пусть выполнено благоприятное событие $\mathcal{E}$, то есть для всех $i,m,t$:

$$
L_{i,m}^{\mathrm{loc}}(t)\le \phi_{i,m}^\star\le U_{i,m}^{\mathrm{loc}}(t).
$$

Тогда для всех $i,\ell,t$:

$$
L_{i,\ell}^{\mathrm{sh}}(t)\le \phi_{i,\ell}^\star\le U_{i,\ell}^{\mathrm{sh}}(t).
$$

Напомним обозначения:

- $i$ — номер ручки;
- $\ell$ — целевое направление;
- $m$ — вспомогательное направление;
- $d_{\ell m}=\|\lambda_\ell-\lambda_m\|_1$;
- $L_s=(1+\rho)R$.


## 19. Доказательство леммы 2

Сначала докажем нижнюю оценку. По определению:

$$
L_{i,\ell}^{\mathrm{sh}}(t)=\max_m\left[L_{i,m}^{\mathrm{loc}}(t)-L_s d_{\ell m}\right].
$$

Зафиксируем произвольное вспомогательное направление $m$. На событии $\mathcal{E}$:

$$
L_{i,m}^{\mathrm{loc}}(t)\le \phi_{i,m}^\star.
$$

По Следствию 1:

$$
|\phi_{i,m}^\star-\phi_{i,\ell}^\star|\le L_s d_{\ell m}.
$$

Отсюда

$$
\phi_{i,m}^\star\le \phi_{i,\ell}^\star+L_s d_{\ell m}.
$$

Значит

$$
L_{i,m}^{\mathrm{loc}}(t)\le \phi_{i,\ell}^\star+L_s d_{\ell m}.
$$

Переносим $L_s d_{\ell m}$ влево:

$$
L_{i,m}^{\mathrm{loc}}(t)-L_s d_{\ell m}\le \phi_{i,\ell}^\star.
$$

Это верно для любого $m$, поэтому верно и для максимума по $m$:

$$
L_{i,\ell}^{\mathrm{sh}}(t)\le \phi_{i,\ell}^\star.
$$

Теперь докажем верхнюю оценку. По определению:

$$
U_{i,\ell}^{\mathrm{sh}}(t)=\min_m\left[U_{i,m}^{\mathrm{loc}}(t)+L_s d_{\ell m}\right].
$$

На событии $\mathcal{E}$:

$$
\phi_{i,m}^\star\le U_{i,m}^{\mathrm{loc}}(t).
$$

По Следствию 1:

$$
\phi_{i,\ell}^\star\le \phi_{i,m}^\star+L_s d_{\ell m}.
$$

Следовательно,

$$
\phi_{i,\ell}^\star\le U_{i,m}^{\mathrm{loc}}(t)+L_s d_{\ell m}.
$$

Это верно для любого $m$, поэтому верно и для минимума по $m$:

$$
\phi_{i,\ell}^\star\le U_{i,\ell}^{\mathrm{sh}}(t).
$$

Лемма доказана.


## 20. Алгоритм LS-CP-FLCB

Полное название алгоритма: **Lipschitz-Shared Constrained Pareto Functional Lower Confidence Bound** 
На каждом шаге алгоритм:

1. хранит общий набор наблюдений $D_i(t)$ для каждой ручки;
2. вычисляет локальные доверительные оценки $L^{\mathrm{loc}}$ и $U^{\mathrm{loc}}$;
3. строит разделяемые оценки $L^{\mathrm{sh}}$ и $U^{\mathrm{sh}}$;
4. выбирает направление с наибольшей неопределенностью;
5. выбирает наиболее перспективную ручку в этом направлении;
6. запускает внутренний оптимизатор;
7. обновляет найденный фронт;
8. исключает заведомо неперспективные ячейки.


## 21. Псевдокод алгоритма

```text
Algorithm LS-CP-FLCB

Input:
    число ручек K
    направления Λ = {λ_1,...,λ_L}
    скаляризация s_λ
    ограничение B
    константа липшицевости L_s
    параметр точности ε
    уровень ошибки δ
    бюджет T

Initialize:
    D_i ← ∅ для всех i
    n[i,l] ← 0 для всех i,l
    active[i,l] ← True для всех i,l
    P_hat ← ∅

For t = 1,...,T:

    1. Для всех i,l вычислить L_loc[i,l] и U_loc[i,l].

    2. Для всех i,l вычислить:

       L_sh[i,l] = max_m (L_loc[i,m] - L_s * ||λ_l - λ_m||_1)
       U_sh[i,l] = min_m (U_loc[i,m] + L_s * ||λ_l - λ_m||_1)

    3. Для каждого направления l вычислить неопределенность:

       W[l] = min_i U_sh[i,l] - min_i L_sh[i,l]

    4. Выбрать направление:

       l_t = argmax_l W[l]

    5. Выбрать ручку:

       i_t = argmin_{i: active[i,l_t]} L_sh[i,l_t]

    6. Запустить внутренний оптимизатор ручки i_t в направлении λ_{l_t}.

    7. Получить x_t, Y_{i_t}(x_t), C_{i_t}(x_t).

    8. Добавить наблюдение в D_{i_t}.

    9. n[i_t,l_t] ← n[i_t,l_t] + 1.

   10. Обновить P_hat по всем найденным допустимым точкам.

   11. Исключить все ячейки (i,l), для которых:

       L_sh[i,l] > min_j U_sh[j,l] + ε.

Output:
    найденный фронт P_hat
    ручки, представленные на P_hat
```


## 22. Правило исключения ячеек

Ячейка $(i,\ell)$ исключается, если

$$
L_{i,\ell}^{\mathrm{sh}}(t)>\min_{j=1,\dots,K}U_{j,\ell}^{\mathrm{sh}}(t)+\varepsilon.
$$

Интерпретация:

- $L_{i,\ell}^{\mathrm{sh}}(t)$ — оптимистическая оценка ячейки $(i,\ell)$;
- $\min_j U_{j,\ell}^{\mathrm{sh}}(t)$ — лучшее подтвержденное значение среди всех ручек в направлении $\ell$;
- если даже оптимистическая оценка ручки $i$ хуже конкурента, то ячейка $(i,\ell)$ не может быть полезной.


## 23. Теорема 1: отсутствие ложного исключения

### Формулировка

Пусть выполнено событие $\mathcal{E}$. Если ячейка $(i,\ell)$ является $\varepsilon$-оптимальной, то есть

$$
\phi_{i,\ell}^\star\le \phi_\ell^\star+\varepsilon,
$$

где

$$
\phi_\ell^\star=\min_j\phi_{j,\ell}^\star,
$$

то алгоритм не исключит ячейку $(i,\ell)$ правилом

$$
L_{i,\ell}^{\mathrm{sh}}(t)>\min_j U_{j,\ell}^{\mathrm{sh}}(t)+\varepsilon.
$$


## 24. Доказательство теоремы 1

По Лемме 2:

$$
L_{i,\ell}^{\mathrm{sh}}(t)\le \phi_{i,\ell}^\star.
$$

Так как ячейка $(i,\ell)$ является $\varepsilon$-оптимальной,

$$
\phi_{i,\ell}^\star\le \phi_\ell^\star+\varepsilon.
$$

Следовательно,

$$
L_{i,\ell}^{\mathrm{sh}}(t)\le \phi_\ell^\star+\varepsilon.
$$

Теперь рассмотрим произвольную ручку $j$. По определению глобального оптимума в направлении $\ell$:

$$
\phi_\ell^\star\le \phi_{j,\ell}^\star.
$$

По Лемме 2:

$$
\phi_{j,\ell}^\star\le U_{j,\ell}^{\mathrm{sh}}(t).
$$

Значит для любого $j$ выполнено

$$
\phi_\ell^\star\le U_{j,\ell}^{\mathrm{sh}}(t).
$$

Следовательно,

$$
\phi_\ell^\star\le \min_j U_{j,\ell}^{\mathrm{sh}}(t).
$$

Тогда

$$
L_{i,\ell}^{\mathrm{sh}}(t)\le \min_j U_{j,\ell}^{\mathrm{sh}}(t)+\varepsilon.
$$

Это противоположно условию исключения. Поэтому ячейка $(i,\ell)$ не исключается.

Теорема доказана.


## 25. Теорема 2: исключение плохих ячеек

### Формулировка

Пусть ячейка $(i,\ell)$ имеет положительный зазор

$$
\Delta_{i,\ell}=\phi_{i,\ell}^\star-\phi_\ell^\star>\varepsilon.
$$

Пусть $j^\star$ — оптимальная ручка в направлении $\ell$:

$$
j^\star\in\arg\min_j\phi_{j,\ell}^\star.
$$

Введем ошибки разделяемых оценок:

$$
\omega_{i,\ell}(t)=\phi_{i,\ell}^\star-L_{i,\ell}^{\mathrm{sh}}(t),
$$

$$
\omega_{\star,\ell}(t)=U_{j^\star,\ell}^{\mathrm{sh}}(t)-\phi_\ell^\star.
$$

Если

$$
\omega_{i,\ell}(t)+\omega_{\star,\ell}(t)<\Delta_{i,\ell}-\varepsilon,
$$

то ячейка $(i,\ell)$ будет исключена.


## 26. Доказательство теоремы 2

По определению $\omega_{i,\ell}(t)$ имеем

$$
L_{i,\ell}^{\mathrm{sh}}(t)=\phi_{i,\ell}^\star-\omega_{i,\ell}(t).
$$

По определению $\omega_{\star,\ell}(t)$ имеем

$$
U_{j^\star,\ell}^{\mathrm{sh}}(t)=\phi_\ell^\star+\omega_{\star,\ell}(t).
$$

Рассмотрим разность:

$$
L_{i,\ell}^{\mathrm{sh}}(t)-U_{j^\star,\ell}^{\mathrm{sh}}(t)
=
\phi_{i,\ell}^\star-\phi_\ell^\star-\omega_{i,\ell}(t)-\omega_{\star,\ell}(t).
$$

Так как

$$
\Delta_{i,\ell}=\phi_{i,\ell}^\star-\phi_\ell^\star,
$$

получаем

$$
L_{i,\ell}^{\mathrm{sh}}(t)-U_{j^\star,\ell}^{\mathrm{sh}}(t)
=
\Delta_{i,\ell}-\omega_{i,\ell}(t)-\omega_{\star,\ell}(t).
$$

По условию теоремы:

$$
\omega_{i,\ell}(t)+\omega_{\star,\ell}(t)<\Delta_{i,\ell}-\varepsilon.
$$

Следовательно,

$$
L_{i,\ell}^{\mathrm{sh}}(t)-U_{j^\star,\ell}^{\mathrm{sh}}(t)>\varepsilon.
$$

То есть

$$
L_{i,\ell}^{\mathrm{sh}}(t)>U_{j^\star,\ell}^{\mathrm{sh}}(t)+\varepsilon.
$$

Так как

$$
\min_j U_{j,\ell}^{\mathrm{sh}}(t)\le U_{j^\star,\ell}^{\mathrm{sh}}(t),
$$

то

$$
L_{i,\ell}^{\mathrm{sh}}(t)>\min_j U_{j,\ell}^{\mathrm{sh}}(t)+\varepsilon.
$$

Следовательно, правило исключения выполнено, и ячейка $(i,\ell)$ исключается.

Теорема доказана.


## 27. Эффективная неопределенность за счет соседних направлений

В независимой версии неопределенность ячейки $(i,\ell)$ уменьшается только при прямых запусках этой ячейки.

В LS-CP-FLCB неопределенность может уменьшаться и за счет соседних направлений. Введем величину

$$
r_{i,\ell}(t)=\min_{m=1,\dots,L}\left[\gamma_{i,m}(n_{i,m}(t))+b_{i,m}(t,\delta)+L_s d_{\ell m}\right].
$$

Здесь:

- $m$ — вспомогательное направление;
- $\gamma_{i,m}$ — ошибка внутренней оптимизации;
- $b_{i,m}$ — статистическая неопределенность;
- $L_s d_{\ell m}$ — цена переноса информации из направления $m$ в направление $\ell$.

Чем меньше $r_{i,\ell}(t)$, тем лучше ячейка $(i,\ell)$ изучена с учетом соседних направлений.


## 28. Следствие: sample complexity для исключения

Если существует направление $m$, близкое к $\ell$, такое что

$$
L_s d_{\ell m}\le \frac{\Delta_{i,\ell}-\varepsilon}{4},
$$

и при некотором $n$ выполнено

$$
\gamma_{i,m}(n)+b_{i,m}(n,\delta)\le \frac{\Delta_{i,\ell}-\varepsilon}{4},
$$

то информация из направления $m$ уже существенно помогает исключить плохую ячейку $(i,\ell)$.

Если

$$
\gamma_{i,m}(n)=a_{i,m}n^{-\alpha_{i,m}},
$$

а доверительный радиус имеет порядок

$$
b_{i,m}(n,\delta)=O\left(\sqrt{\frac{\log(KL/\delta)}{n}}\right),
$$

то достаточно

$$
n=O\left(
\left(\frac{a_{i,m}}{\Delta_{i,\ell}-\varepsilon}\right)^{1/\alpha_{i,m}}
+
\frac{\log(KL/\delta)}{(\Delta_{i,\ell}-\varepsilon)^2}
\right).
$$

Главное отличие от независимого подхода состоит в том, что это число запусков может быть накоплено не в самой ячейке $(i,\ell)$, а в соседней ячейке $(i,m)$.


## 29. Аппроксимация всего Парето-фронта

Для связи скаляризованных задач с аппроксимацией Парето-фронта нужно предположение о покрытии фронта направлениями.

### A5. Сетка направлений

Набор направлений $\Lambda$ является $\eta$-сеткой на симплексе, если для любого $\lambda\in\Delta^{D-1}$ существует $\lambda_\ell\in\Lambda$, такое что

$$
\|\lambda-\lambda_\ell\|_1\le \eta.
$$

### A6. Устойчивость скаляризации

Предположим, что если для всех направлений сетки найдены $\varepsilon_s$-оптимальные допустимые точки, то они дают $\eta_{\mathrm{front}}$-аппроксимацию истинного фронта, где

$$
\eta_{\mathrm{front}}=\zeta(\eta)+\psi(\varepsilon_s).
$$

Здесь:

- $\zeta(\eta)$ — ошибка дискретизации направлений;
- $\psi(\varepsilon_s)$ — ошибка из-за неточного решения скаляризованных задач.


## 30. Теорема 3: аппроксимация Парето-фронта

### Формулировка

Пусть выполнены предположения A1--A6. Пусть для каждого направления $\lambda_\ell\in\Lambda$ алгоритм нашел допустимую точку $\hat y_\ell$, такую что

$$
s_{\lambda_\ell}(\hat y_\ell)\le \phi_\ell^\star+\varepsilon_s.
$$

Тогда найденный фронт

$$
\widehat{\mathcal{P}}_T=\operatorname{ND}(\{\hat y_\ell\}_{\ell=1}^L)
$$

является $\eta_{\mathrm{front}}$-аппроксимацией истинного фронта, то есть для любой точки $p\in\mathcal{P}_{\mathrm{feas}}^\star$ существует $\hat p\in\widehat{\mathcal{P}}_T$, такая что

$$
\|p-\hat p\|_\infty\le \eta_{\mathrm{front}},
$$

где

$$
\eta_{\mathrm{front}}=\zeta(\eta)+\psi(\varepsilon_s).
$$


## 31. Доказательство теоремы 3

Возьмем произвольную точку истинного фронта

$$
p\in\mathcal{P}_{\mathrm{feas}}^\star.
$$

По предположению A6 существует направление $\lambda$ и соответствующая оптимальная скаляризованная точка $y_\lambda^\star$, такая что ошибка представления точки $p$ через это направление контролируется величиной $\zeta(\eta)$.

Так как $\Lambda$ является $\eta$-сеткой, существует направление $\lambda_\ell\in\Lambda$, близкое к $\lambda$:

$$
\|\lambda-\lambda_\ell\|_1\le \eta.
$$

Ошибка, возникающая из-за конечной сетки направлений, не превосходит $\zeta(\eta)$.

По условию теоремы алгоритм нашел точку $\hat y_\ell$, такую что

$$
s_{\lambda_\ell}(\hat y_\ell)\le \phi_\ell^\star+\varepsilon_s.
$$

По предположению устойчивости скаляризации такая $\varepsilon_s$-оптимальность дает ошибку не более $\psi(\varepsilon_s)$ в пространстве целей.

Следовательно, суммарная ошибка покрытия точки $p$ не превосходит

$$
\zeta(\eta)+\psi(\varepsilon_s).
$$

То есть существует найденная точка $\hat p$, такая что

$$
\|p-\hat p\|_\infty\le \zeta(\eta)+\psi(\varepsilon_s).
$$

Удаление доминируемых найденных точек при переходе к $\operatorname{ND}$ не ухудшает покрытие фронта, поскольку доминирующая точка не хуже доминируемой по всем критериям.

Теорема доказана.


## 32. Следствие для IGD

Если для любой точки $p\in\mathcal{P}_{\mathrm{feas}}^\star$ существует точка $\hat p\in\widehat{\mathcal{P}}_T$, такая что

$$
\|p-\hat p\|_\infty\le \eta_{\mathrm{front}},
$$

то для IGD в норме $\ell_\infty$ выполнено

$$
IGD_\infty(\widehat{\mathcal{P}}_T,\mathcal{P}_{\mathrm{feas}}^\star)\le \eta_{\mathrm{front}}.
$$

Для евклидовой нормы имеем

$$
\|p-\hat p\|_2\le \sqrt{D}\|p-\hat p\|_\infty.
$$

Следовательно,

$$
IGD_2(\widehat{\mathcal{P}}_T,\mathcal{P}_{\mathrm{feas}}^\star)
\le
\sqrt{D}\eta_{\mathrm{front}}.
$$


## 33. Следствие для hypervolume gap

Пусть все точки фронта лежат в коробке $[0,R]^D$. Если найденный фронт покрывает истинный фронт с ошибкой $\eta_{\mathrm{front}}$ в норме $\ell_\infty$, то при фиксированной опорной точке можно получить грубую оценку

$$
HV(\mathcal{P}_{\mathrm{feas}}^\star)-HV(\widehat{\mathcal{P}}_T)
\le
D R^{D-1}\eta_{\mathrm{front}}.
$$

Следовательно,

$$
HV\text{-gap}(T)
\le
D R^{D-1}\left(\zeta(\eta)+\psi(\varepsilon_s)\right).
$$

А для hypervolume ratio:

$$
HV\text{-ratio}(T)
\ge
1-
\frac{D R^{D-1}\left(\zeta(\eta)+\psi(\varepsilon_s)\right)}{HV(\mathcal{P}_{\mathrm{feas}}^\star)}.
$$

Эта оценка грубая, но она показывает правильную зависимость: чем плотнее сетка направлений и чем точнее решены скаляризованные задачи, тем меньше потеря по гиперобъему.


## 34. Почему алгоритм вменяем

Теоретическая вменяемость алгоритма LS-CP-FLCB состоит в следующей цепочке утверждений.

1. Локальные доверительные оценки корректны с вероятностью не меньше $1-\delta$.
2. Липшицево распространенные доверительные оценки также корректны.
3. $\varepsilon$-оптимальные ячейки не исключаются.
4. Заведомо плохие ячейки исключаются, когда разделяемая неопределенность становится достаточно малой.
5. Наблюдения, полученные в одном направлении, помогают соседним направлениям.
6. Если скаляризованные задачи на сетке направлений решены достаточно точно, найденный фронт аппроксимирует истинный ограниченный Парето-фронт.
7. Качество аппроксимации выражается через IGD и hypervolume gap.


## 35. Рекомендуемые основные метрики в экспериментах

Для данной постановки основными метриками должны быть:

1. $HV\text{-ratio}$ — доля восстановленного гиперобъема;
2. $HV\text{-gap}$ — потеря гиперобъема;
3. $IGD$ — расстояние от истинного фронта до найденного;
4. additive $\varepsilon$-indicator — аддитивный индикатор покрытия;
5. active cell fraction — доля активных ячеек $(i,\ell)$;
6. unsafe pull rate — доля запусков, нарушающих ограничения.

Метрики по ручкам, такие как recall и precision для $I_{\mathrm{CP}}^\star$, лучше считать дополнительными, а не главными. Они полезны для интерпретации, но могут быть малоинформативны, если почти каждая ручка имеет хотя бы одну Парето-оптимальную конфигурацию.


## 36. Список литературы

[1] Dorn Y., Katrutsa A., Latypov I., Soboleva A. *Functional multi-armed bandit and the best function identification problems*. arXiv:2503.00509, 2025.

[2] Kleinberg R., Slivkins A., Upfal E. *Multi-Armed Bandits in Metric Spaces*. arXiv:0809.4882, 2008.

[3] Bubeck S., Munos R., Stoltz G., Szepesvári C. *X-armed bandits*. Journal of Machine Learning Research, 2011.

[4] Crépon E., Garivier A., Koolen W. M. *Sequential Learning of the Pareto Front for Multi-objective Bandits*. arXiv:2501.17513, 2025.

[5] Daulton S., Balandat M., Bakshy E. *Differentiable Expected Hypervolume Improvement for Parallel Multi-Objective Bayesian Optimization*. NeurIPS, 2020.

[6] Daulton S., Eriksson D., Balandat M., Bakshy E. *Multi-Objective Bayesian Optimization over High-Dimensional Search Spaces*. arXiv:2109.10964, 2021.

[7] Deb K. *Multi-Objective Optimization Using Evolutionary Algorithms*. Wiley, 2001.

[8] Zitzler E., Thiele L. *Multiobjective evolutionary algorithms: a comparative case study and the strength Pareto approach*. IEEE Transactions on Evolutionary Computation, 1999.
